# Worker profiling
Read-only analysis of real vLLM worker traces. Use an unprofiled run for performance comparisons.

See [profiling guide](../docs/profiling.md) for capture settings. No remote jobs are launched by this notebook.

In [ ]:
from pathlib import Path
import os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from llm_serving.analysis import discover_runs, load_run, performance_rows
from llm_serving.profiling import trace_rows, kernel_rows

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'experiment_profiles').is_dir())
OUTPUT_ROOT = Path(os.environ.get('LLM_SERVING_OUTPUT_ROOT', PROJECT_ROOT / 'output')).expanduser()
explicit = os.environ.get('LLM_SERVING_PROFILE_RUN')
runs = [load_run(Path(explicit).expanduser())] if explicit else discover_runs(OUTPUT_ROOT)
profile_runs = [r for r in runs if (r['profile'].get('profiling') or {}).get('enabled') or trace_rows(r['run_dir'])]
run = profile_runs[0] if profile_runs else None
if run is None:
    print('No captured runs found. Enable profiling in a diagnostic experiment; no sample timings are substituted.')
else:
    display(pd.DataFrame([{'run_id': r['run_id'], 'status': r['status'], 'run_dir': str(r['run_dir'])} for r in profile_runs]))
    display(pd.json_normalize(run['profile'], sep='.').T.rename(columns={0: 'capture configuration'}))
    print('Selected:', run['run_dir'])
    print('Warnings:', run['warnings'])

## Diagnostic benchmark measurements
These measurements include profiling overhead. The capture settings are requested limits, not proof of how many iterations were recorded.

In [ ]:
diagnostic = [r for r in performance_rows([run], include_profiled=True) if r['profiled']] if run else []
display(pd.DataFrame(diagnostic).reindex(columns=['case_id', 'config', 'workload', 'profiled', 'output_throughput_tok_per_s', 'ttft_p95_ms', 'itl_p95_ms']))

## Trace inventory
Select a worker below. Earlier retry attempts may be incomplete; compare the attempt number with each case's `retries` count.

In [ ]:
traces = trace_rows(run['run_dir']) if run else []
if run:
    display(pd.DataFrame([{k: c.get(k) for k in ['case_id', 'status', 'retries']} for c in run['summary'].get('cases', [])]))
display(pd.DataFrame(traces))
if run and not traces:
    print('No trace artifacts available: inspect server logs and artifact collection status.')

In [ ]:
TRACE_INDEX = 0  # Choose one row from the inventory above.
if traces:
    selected_trace = Path(traces[TRACE_INDEX]['path'])
    try:
        kernels = pd.DataFrame(kernel_rows(selected_trace))
        display(kernels)
        if not kernels.empty:
            chart = kernels.head(12).iloc[::-1]
            fig, ax = plt.subplots(figsize=(11, 6))
            ax.barh([f"GPU {r.device}: {r.kernel[:90]}" for r in chart.itertuples()], chart['total_ms'])
            ax.set_xlabel('Summed kernel duration (ms; overlapping streams can overlap in time)')
            ax.set_title(selected_trace.name)
            plt.tight_layout()
            plt.show()
        else:
            print('No GPU kernel events in this trace; inspect worker selection and capture support.')
    except (ValueError, OSError, EOFError) as exc:
        print(f'Cannot read trace; it may be incomplete: {exc}')
    print('Open in Perfetto:', selected_trace)
    display(Markdown('[Open Perfetto](https://ui.perfetto.dev/) and load the trace above.'))

## Interpret the timeline
Inspect CPU launches and GPU streams together. Look for GPU gaps, copies, synchronization, and expensive attention/matmul kernels. Use vLLM annotations to identify prefill/decode when available. Kernel-time shares are per device and are not GPU utilization or critical-path time. CUDA graphs and fused kernels can obscure high-level operator attribution.